# EN4554 Assignment 2 on CNNs

**Name:** _your name_  **Index No:** _your index number_

All tasks use CIFAR-10. The 50,000 training images are split into **45,000 train** and **5,000 validation** images. The test set is used **only once, in Task 4**.

## Setup

Steps in this section:
1. Import libraries and select the device (GPU if available).
2. Fix random seeds so results are repeatable.
3. Split the training set into 45k train / 5k validation.
4. Compute per-channel mean and std **from the 45k training images only**.
5. Build normalized datasets and data loaders.
6. Sanity-check the data.

In [ ]:
# 1. Imports and device
import random
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# 2. Fix random seeds
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [ ]:
# 3. Load the raw training set and split it into 45k train / 5k validation
DATA_DIR = "./data"
raw_train = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, download=True)

g = torch.Generator().manual_seed(SEED)                 # separate generator -> split never changes
perm = torch.randperm(len(raw_train), generator=g).tolist()

train_idx = perm[:45000]
val_idx   = perm[45000:]
print("Train images:", len(train_idx), "| Validation images:", len(val_idx))

In [ ]:
# 4. Per-channel mean and std from the 45k TRAINING images only
train_images = raw_train.data[train_idx] / 255.0        # shape (45000, 32, 32, 3), values 0-1

mean = train_images.mean(axis=(0, 1, 2))
std  = train_images.std(axis=(0, 1, 2))
print("Mean (R, G, B):", np.round(mean, 4))
print("Std  (R, G, B):", np.round(std, 4))

In [ ]:
# 5. Normalized datasets and data loaders
transform = transforms.Compose([
    transforms.ToTensor(),                               # (3, 32, 32), values 0-1
    transforms.Normalize(mean.tolist(), std.tolist())    # training statistics
])

full_train = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=transform)
test_set   = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform)

train_set = Subset(full_train, train_idx)
val_set   = Subset(full_train, val_idx)

BATCH = 128
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True, num_workers=2,
                          generator=torch.Generator().manual_seed(SEED))
val_loader   = DataLoader(val_set,  batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set, batch_size=BATCH, shuffle=False, num_workers=2)   # used ONLY in Task 4

classes = full_train.classes
print("Classes:", classes)

In [ ]:
# 6. Sanity check
x, y = next(iter(train_loader))
print("Batch shape:", tuple(x.shape))
print(f"Normalized batch -> mean {x.mean().item():.3f}, std {x.std().item():.3f}")

# show 8 training images (undo normalization only for display)
m = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
s = torch.tensor(std,  dtype=torch.float32).view(3, 1, 1)
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i in range(8):
    img = (x[i] * s + m).permute(1, 2, 0).clamp(0, 1)
    axes[i].imshow(img)
    axes[i].set_title(classes[y[i]], fontsize=9)
    axes[i].axis("off")
plt.suptitle("Sample training images")
plt.show()

# class balance in each split
train_counts = np.bincount(np.array(raw_train.targets)[train_idx], minlength=10)
val_counts   = np.bincount(np.array(raw_train.targets)[val_idx],   minlength=10)
print(f"{'Class':<12}{'Train':>8}{'Val':>8}")
for c, tr, va in zip(classes, train_counts, val_counts):
    print(f"{c:<12}{tr:>8}{va:>8}")

**Interpretation (Setup):** _[to be completed after running — see note]_

## Task 1: Backpropagation

### (a) Linear, ReLU and softmax cross-entropy with explicit forward/backward

Each layer has a `forward` method and a `backward` method. No autograd is used.
Every backward follows the chain rule: **downstream gradient = local gradient × upstream gradient**.

| Layer | Forward | Backward (`dout` = upstream gradient) |
|---|---|---|
| Linear | $Y = XW + b$ | $\frac{\partial L}{\partial W} = X^T \, dout$, $\frac{\partial L}{\partial b} = \sum_n dout$, $\frac{\partial L}{\partial X} = dout \, W^T$ |
| ReLU | $Y = \max(0, X)$ | $\frac{\partial L}{\partial X} = dout \cdot \mathbb{1}[X > 0]$ |
| Sigmoid (for part d) | $Y = \sigma(X)$ | $\frac{\partial L}{\partial X} = dout \cdot \sigma(X)(1 - \sigma(X))$ |
| Softmax CE | $p = \text{softmax}(s)$, $L = -\frac{1}{N}\sum_n \log p_{n, y_n}$ | $\frac{\partial L}{\partial s} = \frac{1}{N}(p - \text{onehot}(y))$ |

Weights use He initialization ($\text{std} = \sqrt{2 / D_{in}}$) and biases start at zero.

In [ ]:
# Layers with explicit forward / backward (no autograd)

class Linear:
    def __init__(self, D_in, D_out):
        std = (2.0 / D_in) ** 0.5                         # He initialization
        self.W = torch.randn(D_in, D_out, device=device) * std
        self.b = torch.zeros(D_out, device=device)

    def forward(self, x):
        self.x = x                                        # save input for backward
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = dout.sum(dim=0)
        return dout @ self.W.T                            # gradient for the previous layer


class ReLU:
    def forward(self, x):
        self.mask = (x > 0)
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask


class Sigmoid:
    def forward(self, x):
        self.out = 1.0 / (1.0 + torch.exp(-x))
        return self.out

    def backward(self, dout):
        return dout * self.out * (1 - self.out)


class SoftmaxCrossEntropy:
    def forward(self, scores, y):
        N = scores.shape[0]
        shifted = scores - scores.max(dim=1, keepdim=True).values   # avoid overflow in exp
        exp_s = torch.exp(shifted)
        self.probs = exp_s / exp_s.sum(dim=1, keepdim=True)
        self.y = y
        return -torch.log(self.probs[torch.arange(N), y]).mean()

    def backward(self):
        N = self.probs.shape[0]
        dscores = self.probs.clone()
        dscores[torch.arange(N), self.y] -= 1                       # p - onehot(y)
        return dscores / N

In [ ]:
# Assemble the MLP

class MLP:
    def __init__(self, sizes, activation="relu"):
        # sizes example: [3072, 100, 10] -> Linear, ReLU, Linear
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:                                  # no activation after last layer
                self.layers.append(ReLU() if activation == "relu" else Sigmoid())
        self.loss_fn = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x                                                    # class scores

    def loss(self, x, y):
        return self.loss_fn.forward(self.forward(x), y)

    def backward(self):
        dout = self.loss_fn.backward()
        for layer in reversed(self.layers):
            dout = layer.backward(dout)

    def linear_layers(self):
        return [layer for layer in self.layers if isinstance(layer, Linear)]

In [ ]:
# Quick check: initial loss should be close to the random-guess loss ln(10)
set_seed(SEED)
mlp = MLP([3072, 100, 10])
x, y = next(iter(train_loader))
x = x.view(x.shape[0], -1).to(device)
y = y.to(device)

print(f"Initial loss: {mlp.loss(x, y).item():.4f}   (random guessing gives ln(10) = {np.log(10):.4f})")
mlp.backward()
for i, lin in enumerate(mlp.linear_layers(), 1):
    print(f"Layer {i}: W {tuple(lin.W.shape)}, dW {tuple(lin.dW.shape)}, b {tuple(lin.b.shape)}, db {tuple(lin.db.shape)}")

**Interpretation (a):** _[to be completed after running — see note]_

### (b) Gradient check against autograd and centred finite differences

- A small network (20 → 15 → 10) with 8 random inputs is used, so **every** weight can be checked by finite differences in a short time.
- All values are in **float64**, because finite differences subtract two very close loss values.
- Centred finite difference: $\frac{\partial L}{\partial w} \approx \frac{L(w+h) - L(w-h)}{2h}$, with $h = 10^{-6}$.
- Relative error: $\frac{\lVert a - b \rVert}{\lVert a \rVert + \lVert b \rVert}$.

In [ ]:
# Step 1: gradients from our manual backward pass
set_seed(SEED)
N, D, H, C = 8, 20, 15, 10
x_small = torch.randn(N, D, dtype=torch.float64, device=device)
y_small = torch.randint(0, C, (N,), device=device)

net = MLP([D, H, C])
for lin in net.linear_layers():                                     # float64 + non-zero biases
    lin.W = lin.W.double()
    lin.b = (0.1 * torch.randn(lin.b.shape, device=device)).double()

net.loss(x_small, y_small)
net.backward()
L1, L2 = net.linear_layers()
manual_grads = {"W1": L1.dW, "b1": L1.db, "W2": L2.dW, "b2": L2.db}

In [ ]:
# Step 2: gradients from PyTorch autograd (same weights, same computation)
W1 = L1.W.clone().requires_grad_(True)
b1 = L1.b.clone().requires_grad_(True)
W2 = L2.W.clone().requires_grad_(True)
b2 = L2.b.clone().requires_grad_(True)

scores = torch.relu(x_small @ W1 + b1) @ W2 + b2
loss_autograd = F.cross_entropy(scores, y_small)
loss_autograd.backward()
autograd_grads = {"W1": W1.grad, "b1": b1.grad, "W2": W2.grad, "b2": b2.grad}

In [ ]:
# Step 3: gradients from centred finite differences
def numerical_grad(net, x, y, param, h=1e-6):
    grad = torch.zeros_like(param)
    p_flat, g_flat = param.view(-1), grad.view(-1)
    for i in range(p_flat.numel()):
        old = p_flat[i].item()
        p_flat[i] = old + h
        loss_plus = net.loss(x, y).item()
        p_flat[i] = old - h
        loss_minus = net.loss(x, y).item()
        p_flat[i] = old                                             # restore the weight
        g_flat[i] = (loss_plus - loss_minus) / (2 * h)
    return grad

params = {"W1": L1.W, "b1": L1.b, "W2": L2.W, "b2": L2.b}
numerical_grads = {name: numerical_grad(net, x_small, y_small, p) for name, p in params.items()}

In [ ]:
# Step 4: relative errors and torch.allclose
def rel_error(a, b):
    return (torch.norm(a - b) / (torch.norm(a) + torch.norm(b) + 1e-12)).item()

print(f"{'Param':<7}{'Shape':<10}{'Rel. err (autograd)':>21}{'allclose':>10}"
      f"{'Rel. err (finite diff)':>25}{'allclose':>10}")
print("-" * 83)
for name in manual_grads:
    a, ag, nm = manual_grads[name], autograd_grads[name], numerical_grads[name]
    e_ag, ok_ag = rel_error(a, ag), torch.allclose(a, ag, rtol=1e-5, atol=1e-8)
    e_nm, ok_nm = rel_error(a, nm), torch.allclose(a, nm, rtol=1e-5, atol=1e-8)
    print(f"{name:<7}{str(tuple(a.shape)):<10}{e_ag:>21.2e}{str(ok_ag):>10}{e_nm:>25.2e}{str(ok_nm):>10}")

**Interpretation (b):** _[to be completed after running — see note]_

### (c) Training with SGD on flattened CIFAR-10

- Model: 3072 → 100 → 10 (Linear–ReLU–Linear), the fully connected network from the L06 example.
- Each image is flattened from 3 × 32 × 32 to a 3072-vector.
- Plain mini-batch SGD: $W \leftarrow W - \eta \, \partial L / \partial W$, with learning rate 0.01, batch size 128, 10 epochs.
- After each epoch, validation loss and accuracy are measured on the 5k validation set.

In [ ]:
# Helper functions: SGD update and evaluation
def sgd_step(model, lr):
    for lin in model.linear_layers():
        lin.W -= lr * lin.dW
        lin.b -= lr * lin.db

def evaluate(model, loader):
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x = x.view(x.shape[0], -1).to(device)
        y = y.to(device)
        scores = model.forward(x)
        total_loss += model.loss_fn.forward(scores, y).item() * y.shape[0]
        correct += (scores.argmax(dim=1) == y).sum().item()
        total += y.shape[0]
    return total_loss / total, correct / total

In [ ]:
# Training loop
set_seed(SEED)
model = MLP([3072, 100, 10])
LR, EPOCHS = 0.01, 10

iter_losses, train_losses, val_losses, val_accs = [], [], [], []

for epoch in range(EPOCHS):
    running = 0.0
    for x, y in train_loader:
        x = x.view(x.shape[0], -1).to(device)            # flatten
        y = y.to(device)

        loss = model.loss(x, y)                          # forward
        model.backward()                                 # manual backward
        sgd_step(model, LR)                              # update

        iter_losses.append(loss.item())
        running += loss.item()

    train_losses.append(running / len(train_loader))
    v_loss, v_acc = evaluate(model, val_loader)
    val_losses.append(v_loss)
    val_accs.append(v_acc)
    print(f"Epoch {epoch+1:2d} | train loss {train_losses[-1]:.4f} | "
          f"val loss {v_loss:.4f} | val acc {v_acc*100:.2f}%")

print(f"\nFinal validation accuracy: {val_accs[-1]*100:.2f}%")

In [ ]:
# Plots: training loss and validation accuracy
iters_per_epoch = len(train_loader)
epochs_axis = range(1, EPOCHS + 1)

fig, ax = plt.subplots(1, 3, figsize=(17, 4))

ax[0].plot(iter_losses, alpha=0.4, label="per iteration")
ax[0].plot([iters_per_epoch * e for e in epochs_axis], train_losses, "o-", label="epoch average")
ax[0].set_xlabel("Iteration"); ax[0].set_ylabel("Loss")
ax[0].set_title("Training loss"); ax[0].legend()

ax[1].plot(epochs_axis, train_losses, "o-", label="train")
ax[1].plot(epochs_axis, val_losses, "s-", label="validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Loss")
ax[1].set_title("Train vs validation loss"); ax[1].legend()

ax[2].plot(epochs_axis, [a * 100 for a in val_accs], "o-")
ax[2].set_xlabel("Epoch"); ax[2].set_ylabel("Accuracy (%)")
ax[2].set_title("Validation accuracy")

plt.tight_layout()
plt.show()

**Interpretation (c):** _[to be completed after running — see note]_

### (d) Per-layer gradient norm at initialization: sigmoid vs ReLU

- 10-layer MLP: 3072 → 256 (× 9 hidden layers) → 10, i.e. 10 Linear layers.
- Both networks use the **same seed and the same He initialization**, so the only difference is the activation function.
- One forward and one backward pass on a single training batch, **no weight update**.
- For each Linear layer we record $\lVert \partial L / \partial W \rVert$ (Frobenius norm).

In [ ]:
# Gradient norms at initialization
x_d, y_d = next(iter(train_loader))
x_d = x_d.view(x_d.shape[0], -1).to(device)
y_d = y_d.to(device)

sizes = [3072] + [256] * 9 + [10]
grad_norms = {}

for act in ["sigmoid", "relu"]:
    set_seed(SEED)                                       # same initial weights for both
    deep = MLP(sizes, activation=act)
    deep.loss(x_d, y_d)
    deep.backward()
    grad_norms[act] = [torch.norm(lin.dW).item() for lin in deep.linear_layers()]

print(f"{'Layer':<7}{'Sigmoid':>12}{'ReLU':>12}")
for i in range(10):
    print(f"{i+1:<7}{grad_norms['sigmoid'][i]:>12.2e}{grad_norms['relu'][i]:>12.2e}")
print(f"\nRatio layer10/layer1 -> sigmoid: {grad_norms['sigmoid'][9]/grad_norms['sigmoid'][0]:.1e}, "
      f"ReLU: {grad_norms['relu'][9]/grad_norms['relu'][0]:.1e}")

In [ ]:
# Plot (log scale)
layers = range(1, 11)
plt.figure(figsize=(8, 4.5))
plt.semilogy(layers, grad_norms["sigmoid"], "o-", label="Sigmoid")
plt.semilogy(layers, grad_norms["relu"], "s-", label="ReLU")
plt.xlabel("Layer (1 = closest to input, 10 = output)")
plt.ylabel("Gradient norm ||dL/dW|| (log scale)")
plt.title("Per-layer gradient norm at initialization (10-layer MLP)")
plt.xticks(layers)
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

**Interpretation (d):** _[to be completed after running — see note]_